# 01. Fundamentals

## Quick notes
### What this covers
Creating tables, SQLite's flexible type system, inserting rows, basic SELECT, filtering, sorting, paging, DISTINCT, and the order in which SQL really runs.
### Key points
- SQLite has 5 storage classes: NULL, INTEGER, REAL, TEXT, BLOB. A column's declared type is only a hint (its "affinity").
- `typeof(x)` tells you what was really stored.
- Multi-row `INSERT ... VALUES`, `INSERT ... SELECT`, `INSERT OR IGNORE` and `INSERT OR REPLACE` cover most loading jobs.
- Without `ORDER BY`, row order is not promised. Always sort when order matters.
- `LIMIT/OFFSET` gets slow on deep pages; keyset pagination (`WHERE id > last_seen`) stays fast.
- SQL is written `SELECT ... FROM ... WHERE`, but runs `FROM -> WHERE -> GROUP BY -> HAVING -> SELECT -> DISTINCT -> ORDER BY -> LIMIT`.
### Gotchas (the part worth revising twice)
- A TEXT value like `'abc'` can be stored in a REAL column without any error (unless you use STRICT or a CHECK).
- TEXT sorts after every number, so `'abc' > 80000` is true.
- `INSERT OR REPLACE` deletes the old row and inserts a new one: columns you did not list become NULL/default.
- Big `OFFSET` values still read and throw away all the skipped rows.
- You cannot use a SELECT alias inside `WHERE` (SQLite is lenient here, other databases are not). Aliases are fine in `ORDER BY`.
- `ORDER BY` on a non-unique column gives unstable tie order. Add a unique tie-breaker.
### When it matters
Every query you will ever write. Type surprises and unstable paging are the classic production bugs.
### One-line recap example
`SELECT DISTINCT dept FROM employees WHERE salary > 50000 ORDER BY dept LIMIT 5;`

## 01. CREATE TABLE and the five storage classes
- A table has named columns, each with a declared type.
- SQLite stores every value as one of: NULL, INTEGER, REAL, TEXT, BLOB.
- `INTEGER PRIMARY KEY` makes the column an alias of the hidden `rowid` (see section 16).
- Seed data used through this notebook is created here.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE employees (
    emp_id      INTEGER PRIMARY KEY,
    first_name  TEXT NOT NULL,
    last_name   TEXT NOT NULL,
    dept        TEXT,
    salary      REAL,
    photo       BLOB,
    notes       TEXT
);

INSERT INTO employees (emp_id, first_name, last_name, dept, salary, notes) VALUES
    (1, 'Alice', 'Nguyen', 'Engineering', 95000.50, 'Team lead'),
    (2, 'Bob',   'Smith',  'Sales',       72000.00, NULL),
    (3, 'Carla', 'Diaz',   'Engineering', 81000.75, 'Remote'),
    (4, 'dev',   'Roy',    'Sales',       'not-a-number', NULL),
    (5, 'Eve',   'Owusu',  NULL,          NULL, NULL);

SELECT emp_id, first_name, salary, typeof(salary) AS salary_class FROM employees ORDER BY emp_id;

emp_id | first_name | salary       | salary_class
-------+------------+--------------+-------------
1      | Alice      | 95000.5      | real        
2      | Bob        | 72000.0      | real        
3      | Carla      | 81000.75     | real        
4      | dev        | not-a-number | text        
5      | Eve        | NULL         | null        
(5 rows)



## 02. Type affinity: the declared type is a hint
- Affinity is chosen from the declared type name by simple rules:
  contains `INT` -> INTEGER; contains `CHAR`, `CLOB`, `TEXT` -> TEXT; contains `BLOB` or empty -> BLOB (none);
  contains `REAL`, `FLOA`, `DOUB` -> REAL; anything else (e.g. `DECIMAL`, `BOOLEAN`, `DATE`) -> NUMERIC.
- On insert, SQLite tries to convert the value to the column's affinity *only if the conversion is lossless*.
- `'42'` into an INTEGER column becomes the integer 42. `'abc'` stays text.
- `DATE`, `BOOLEAN`, `VARCHAR(50)` do not enforce anything. `VARCHAR(5)` happily stores 20 characters.

In [3]:
%%sql
CREATE TABLE affinity_demo (
    i  INTEGER,
    r  REAL,
    t  TEXT,
    n  NUMERIC,
    b  BLOB,
    v  VARCHAR(3)
);
INSERT INTO affinity_demo VALUES ('42', '3.5', 42, '7.0', '42', 'way too long for three');
INSERT INTO affinity_demo VALUES ('42abc', 'x', 3.5, 'x', 42, 'ok');
SELECT typeof(i) i, typeof(r) r, typeof(t) t, typeof(n) n, typeof(b) b, v
FROM affinity_demo ORDER BY rowid;

i       | r    | t    | n       | b       | v                     
--------+------+------+---------+---------+-----------------------
integer | real | text | integer | text    | way too long for three
text    | text | text | text    | integer | ok                    
(2 rows)



## 03. typeof() and mixed values in one column
- `typeof(x)` returns `'null'`, `'integer'`, `'real'`, `'text'` or `'blob'`.
- One column can hold different classes in different rows.
- `STRICT` tables (SQLite 3.37+) refuse wrong types. Compare below.
- Use `typeof` to audit data that came in through a loose import.

In [4]:
%%sql
SELECT typeof(1), typeof(1.0), typeof('1'), typeof(x'01'), typeof(NULL), typeof(1+1.0), typeof(10/4), typeof(10/4.0);

typeof(1) | typeof(1.0) | typeof('1') | typeof(x'01') | typeof(NULL) | typeof(1+1.0) | typeof(10/4) | typeof(10/4.0)
----------+-------------+-------------+---------------+--------------+---------------+--------------+---------------
integer   | real        | text        | blob          | null         | real          | integer      | real          
(1 row)



In [5]:
%%sql expect_error
CREATE TABLE strict_demo (id INTEGER PRIMARY KEY, qty INTEGER NOT NULL) STRICT;
INSERT INTO strict_demo VALUES (1, 5);
INSERT INTO strict_demo VALUES (2, 'five');
SELECT * FROM strict_demo ORDER BY id;

ERROR: cannot store TEXT value in INTEGER column strict_demo.qty

id | qty
---+----
1  | 5  
(1 row)



In [6]:
%%sql
-- audit: find the bad value in the loose table
SELECT emp_id, salary FROM employees WHERE typeof(salary) NOT IN ('real', 'integer', 'null') ORDER BY emp_id;

emp_id | salary      
-------+-------------
4      | not-a-number
(1 row)



## 04. INSERT: single, multi-row, partial columns
- Always list the column names: `INSERT INTO t (a, b) VALUES (...)`. It survives schema changes.
- One multi-row `VALUES` is a single statement: faster and atomic.
- Unlisted columns get their DEFAULT, or NULL when there is no default.

In [7]:
%%sql
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name       TEXT NOT NULL UNIQUE,
    category   TEXT NOT NULL DEFAULT 'misc',
    price      REAL NOT NULL CHECK (price >= 0),
    stock      INTEGER NOT NULL DEFAULT 0
);
INSERT INTO products (name, category, price, stock) VALUES
    ('Keyboard', 'tech', 49.90, 25),
    ('Mouse',    'tech', 19.50, 40),
    ('Notebook', 'office', 3.20, 300);
INSERT INTO products (name, price) VALUES ('Mystery box', 9.99);   -- category and stock use defaults
SELECT * FROM products ORDER BY product_id;

product_id | name        | category | price | stock
-----------+-------------+----------+-------+------
1          | Keyboard    | tech     | 49.9  | 25   
2          | Mouse       | tech     | 19.5  | 40   
3          | Notebook    | office   | 3.2   | 300  
4          | Mystery box | misc     | 9.99  | 0    
(4 rows)



## 05. INSERT ... SELECT and DEFAULT VALUES
- `INSERT INTO target SELECT ... FROM source` copies rows in one statement. Good for archives and backfills.
- `INSERT INTO t DEFAULT VALUES` inserts one row using only defaults.
- Column count and order of the SELECT must match the insert column list.

In [8]:
%%sql
CREATE TABLE expensive_products (name TEXT, price REAL);
INSERT INTO expensive_products (name, price)
SELECT name, price FROM products WHERE price > 10 ORDER BY name;
SELECT * FROM expensive_products ORDER BY name;

name     | price
---------+------
Keyboard | 49.9 
Mouse    | 19.5 
(2 rows)



In [9]:
%%sql
CREATE TABLE audit_log (id INTEGER PRIMARY KEY, kind TEXT DEFAULT 'info', note TEXT);
INSERT INTO audit_log DEFAULT VALUES;
SELECT * FROM audit_log ORDER BY id;

id | kind | note
---+------+-----
1  | info | NULL
(1 row)



## 06. INSERT OR IGNORE / OR REPLACE (conflict handling)
- `UNIQUE`/`PRIMARY KEY` clashes normally raise an error.
- `INSERT OR IGNORE` silently skips the clashing row.
- `INSERT OR REPLACE` (same as `REPLACE INTO`) deletes the old row, then inserts the new one.
- Gotcha: REPLACE is delete + insert. Columns you did not mention reset to their defaults, and the old row's other data is lost.
- Bigger story (UPSERT, `ON CONFLICT DO UPDATE`): see 06-Data-Modification/01-Data-Modification.

In [10]:
%%sql expect_error
INSERT INTO products (name, price) VALUES ('Mouse', 25.00);

ERROR: UNIQUE constraint failed: products.name



In [11]:
%%sql
INSERT OR IGNORE INTO products (name, price) VALUES ('Mouse', 25.00);
SELECT name, category, price, stock FROM products WHERE name = 'Mouse';

name  | category | price | stock
------+----------+-------+------
Mouse | tech     | 19.5  | 40   
(1 row)



In [12]:
%%sql
-- bad: REPLACE wipes the unlisted columns (stock was 40, category was 'tech') and the row gets a new product_id (was 2)
INSERT OR REPLACE INTO products (name, price) VALUES ('Mouse', 25.00);
SELECT product_id, name, category, price, stock FROM products ORDER BY product_id;

product_id | name        | category | price | stock
-----------+-------------+----------+-------+------
1          | Keyboard    | tech     | 49.9  | 25   
3          | Notebook    | office   | 3.2   | 300  
4          | Mystery box | misc     | 9.99  | 0    
5          | Mouse       | misc     | 25.0  | 0    
(4 rows)



## 07. Basic SELECT and column order
- `SELECT col1, col2 FROM t` returns exactly those columns in exactly that order.
- `SELECT *` returns columns in table-definition order. Avoid it in application code: adding a column changes what you get.
- Column order in the result is whatever you wrote, not how the table was built.

In [13]:
%%sql
SELECT name, price, category FROM products ORDER BY name;

name        | price | category
------------+-------+---------
Keyboard    | 49.9  | tech    
Mouse       | 25.0  | misc    
Mystery box | 9.99  | misc    
Notebook    | 3.2   | office  
(4 rows)



In [14]:
%%sql
SELECT * FROM products ORDER BY name LIMIT 2;

product_id | name     | category | price | stock
-----------+----------+----------+-------+------
1          | Keyboard | tech     | 49.9  | 25   
5          | Mouse    | misc     | 25.0  | 0    
(2 rows)



## 08. Expressions and aliases
- A SELECT list can hold math, functions, string joins and constants, not just columns.
- `AS alias` names the result column. Quote it with `"..."` if it has spaces.
- Aliases can be used in `ORDER BY`, but not reliably in `WHERE` (see section 17).
- Expressions with no FROM are fine: `SELECT 1 + 1;`.

In [15]:
%%sql
SELECT name,
       price,
       stock,
       price * stock                AS inventory_value,
       round(price * 1.18, 2)       AS price_with_tax,
       upper(name) || ' (' || category || ')' AS label,
       'USD'                        AS currency
FROM products
ORDER BY inventory_value DESC, name;

name        | price | stock | inventory_value | price_with_tax | label              | currency
------------+-------+-------+-----------------+----------------+--------------------+---------
Keyboard    | 49.9  | 25    | 1247.5          | 58.88          | KEYBOARD (tech)    | USD     
Notebook    | 3.2   | 300   | 960.0           | 3.78           | NOTEBOOK (office)  | USD     
Mouse       | 25.0  | 0     | 0.0             | 29.5           | MOUSE (misc)       | USD     
Mystery box | 9.99  | 0     | 0.0             | 11.79          | MYSTERY BOX (misc) | USD     
(4 rows)



In [16]:
%%sql
SELECT 1 + 1 AS two, 7 / 2 AS int_div, 7 / 2.0 AS real_div, 'a' || 'b' AS joined, length('hello') AS len;

two | int_div | real_div | joined | len
----+---------+----------+--------+----
2   | 3       | 3.5      | ab     | 5  
(1 row)



## 09. WHERE: filter rows before they are returned
- `WHERE` keeps only rows where the condition is TRUE. FALSE and NULL rows are dropped.
- It sees raw column values, so type-affinity comparison rules apply.
- Deeper operators (LIKE, IN, BETWEEN, NULL logic): see 02-Filtering-And-Operators.

In [17]:
%%sql
SELECT first_name, salary FROM employees WHERE salary > 80000 ORDER BY emp_id;

first_name | salary      
-----------+-------------
Alice      | 95000.5     
Carla      | 81000.75    
dev        | not-a-number
(3 rows)



- Gotcha above: `'not-a-number' > 80000` is true. SQLite ranks all TEXT above all numbers. Guard it:

In [18]:
%%sql
SELECT first_name, salary FROM employees
WHERE typeof(salary) IN ('real', 'integer') AND salary > 80000
ORDER BY emp_id;

first_name | salary  
-----------+---------
Alice      | 95000.5 
Carla      | 81000.75
(2 rows)



## 10. ORDER BY: direction and multiple keys
- Default is `ASC`. `DESC` reverses.
- Several keys: sort by the first, break ties with the next, and so on. Each key has its own direction.
- You can sort by an alias, an expression, or a column position (`ORDER BY 2`; fragile, avoid).
- Add a unique column (like the id) as the last key to make the order fully deterministic.

In [19]:
%%sql
SELECT dept, last_name, salary FROM employees
ORDER BY dept ASC, salary DESC, emp_id;

dept        | last_name | salary      
------------+-----------+-------------
NULL        | Owusu     | NULL        
Engineering | Nguyen    | 95000.5     
Engineering | Diaz      | 81000.75    
Sales       | Roy       | not-a-number
Sales       | Smith     | 72000.0     
(5 rows)



## 11. NULLs in ORDER BY: NULLS FIRST / NULLS LAST
- In SQLite, NULL counts as the smallest value: first in `ASC`, last in `DESC`.
- `NULLS FIRST` / `NULLS LAST` (SQLite 3.30+) overrides that.
- Other databases may place NULLs the opposite way. State it explicitly when it matters.

In [20]:
%%sql
SELECT first_name, dept FROM employees ORDER BY dept, emp_id;            -- NULL first (default for ASC)

first_name | dept       
-----------+------------
Eve        | NULL       
Alice      | Engineering
Carla      | Engineering
Bob        | Sales      
dev        | Sales      
(5 rows)



In [21]:
%%sql
SELECT first_name, dept FROM employees ORDER BY dept NULLS LAST, emp_id;

first_name | dept       
-----------+------------
Alice      | Engineering
Carla      | Engineering
Bob        | Sales      
dev        | Sales      
Eve        | NULL       
(5 rows)



In [22]:
%%sql
SELECT first_name, dept FROM employees ORDER BY dept DESC NULLS FIRST, emp_id;

first_name | dept       
-----------+------------
Eve        | NULL       
Bob        | Sales      
dev        | Sales      
Alice      | Engineering
Carla      | Engineering
(5 rows)



## 12. COLLATE: case-insensitive sorting
- Default text comparison is `BINARY`: byte by byte, so every uppercase letter sorts before every lowercase one.
- `COLLATE NOCASE` ignores case (ASCII letters only).
- `COLLATE` can be set on the column definition, or per query in `ORDER BY` / `WHERE`.
- `COLLATE NOCASE` in WHERE makes `=` case-insensitive too.

In [23]:
%%sql
SELECT first_name FROM employees ORDER BY first_name;                       -- 'dev' lands last

first_name
----------
Alice     
Bob       
Carla     
Eve       
dev       
(5 rows)



In [24]:
%%sql
SELECT first_name FROM employees ORDER BY first_name COLLATE NOCASE;        -- 'dev' sorts as 'D'

first_name
----------
Alice     
Bob       
Carla     
dev       
Eve       
(5 rows)



In [25]:
%%sql
SELECT first_name FROM employees WHERE first_name = 'DEV' COLLATE NOCASE;

first_name
----------
dev       
(1 row)



## 13. LIMIT and OFFSET (and why order matters)
- `LIMIT n` caps the rows. `OFFSET k` skips k rows first.
- Page p of size s: `LIMIT s OFFSET s * (p - 1)`.
- Without `ORDER BY` the "first" rows are arbitrary. Paging without a stable order can repeat or miss rows.
- Ties are the usual cause: add a unique tie-breaker to the ORDER BY.

In [26]:
%%sql
CREATE TABLE articles (id INTEGER PRIMARY KEY, title TEXT, published TEXT);
INSERT INTO articles (title, published) VALUES
    ('A', '2024-01-01'), ('B', '2024-01-01'), ('C', '2024-01-02'), ('D', '2024-01-02'),
    ('E', '2024-01-03'), ('F', '2024-01-03'), ('G', '2024-01-04'), ('H', '2024-01-05');
-- page 2 (size 3): ordered by id, stable
SELECT id, title FROM articles ORDER BY id LIMIT 3 OFFSET 3;

id | title
---+------
4  | D    
5  | E    
6  | F    
(3 rows)



In [27]:
%%sql
-- good: ties on `published` broken by id, so every page is repeatable
SELECT id, title, published FROM articles ORDER BY published DESC, id DESC LIMIT 3 OFFSET 0;

id | title | published 
---+-------+-----------
8  | H     | 2024-01-05
7  | G     | 2024-01-04
6  | F     | 2024-01-03
(3 rows)



In [28]:
%%sql
SELECT id, title, published FROM articles ORDER BY published DESC, id DESC LIMIT 3 OFFSET 3;

id | title | published 
---+-------+-----------
5  | E     | 2024-01-03
4  | D     | 2024-01-02
3  | C     | 2024-01-02
(3 rows)



In [29]:
%%sql
-- SQLite also accepts LIMIT offset, count (note the reversed order!)
SELECT id, title FROM articles ORDER BY id LIMIT 3, 2;

id | title
---+------
4  | D    
5  | E    
(2 rows)



## 14. Pagination gotchas: OFFSET cost and keyset pagination
- `OFFSET 100000` makes the database walk and discard 100000 rows before returning any. Deep pages get slower and slower.
- Inserts or deletes between page loads shift rows, so a user may see a row twice or miss one.
- Keyset (seek) pagination: remember the last row's sort key and ask for rows after it. It uses an index, stays fast on any page, and is stable.
- Needs a unique, indexed sort key. For non-unique sorts use a tuple: `WHERE (published, id) < (?, ?)`.

In [30]:
%%sql
-- page 1
SELECT id, title FROM articles ORDER BY id LIMIT 3;

id | title
---+------
1  | A    
2  | B    
3  | C    
(3 rows)



In [31]:
%%sql
-- page 2: client sends last_seen_id = 3
SELECT id, title FROM articles WHERE id > 3 ORDER BY id LIMIT 3;

id | title
---+------
4  | D    
5  | E    
6  | F    
(3 rows)



In [32]:
%%sql
-- two-column keyset: after row (published='2024-01-03', id=6), newest-first
SELECT id, title, published FROM articles
WHERE (published, id) < ('2024-01-03', 6)
ORDER BY published DESC, id DESC LIMIT 3;

id | title | published 
---+-------+-----------
5  | E     | 2024-01-03
4  | D     | 2024-01-02
3  | C     | 2024-01-02
(3 rows)



## 15. DISTINCT
- Removes duplicate rows from the result. It compares the whole selected row, not one column.
- NULLs count as equal to each other for DISTINCT (one NULL row survives).
- `DISTINCT` is often a smell: if you needed it to hide join duplicates, fix the join. See 02-Querying/01-Joins.
- `SELECT DISTINCT a, b` returns unique pairs, not unique a's.

In [33]:
%%sql
SELECT dept FROM employees ORDER BY dept, emp_id;

dept       
-----------
NULL       
Engineering
Engineering
Sales      
Sales      
(5 rows)



In [34]:
%%sql
SELECT DISTINCT dept FROM employees ORDER BY dept;

dept       
-----------
NULL       
Engineering
Sales      
(3 rows)



In [35]:
%%sql
-- unique pairs
SELECT DISTINCT dept, typeof(salary) AS salary_class FROM employees ORDER BY dept, salary_class;

dept        | salary_class
------------+-------------
NULL        | null        
Engineering | real        
Sales       | real        
Sales       | text        
(4 rows)



In [36]:
%%sql
-- count distinct values (NULLs are ignored by COUNT(col))
SELECT count(*) AS rows_, count(dept) AS non_null, count(DISTINCT dept) AS distinct_depts FROM employees;

rows_ | non_null | distinct_depts
------+----------+---------------
5     | 4        | 2             
(1 row)



## 16. rowid: the hidden key
- Every ordinary SQLite table has a hidden 64-bit integer `rowid`.
- A column declared exactly `INTEGER PRIMARY KEY` is an alias for it. It auto-assigns max(rowid)+1 when you leave it out.
- `INTEGER PRIMARY KEY AUTOINCREMENT` additionally guarantees ids are never reused (small extra cost). Plain is usually enough.
- `INT PRIMARY KEY` is NOT an alias (only the exact word INTEGER is). Classic trap.
- Tables created `WITHOUT ROWID` have no hidden rowid.
- `last_insert_rowid()` returns the id of the latest insert on this connection.

In [37]:
%%sql
SELECT rowid AS row_id, emp_id, first_name FROM employees ORDER BY rowid LIMIT 3;   -- same numbers: emp_id is the rowid

row_id | emp_id | first_name
-------+--------+-----------
1      | 1      | Alice     
2      | 2      | Bob       
3      | 3      | Carla     
(3 rows)



In [38]:
%%sql
INSERT INTO employees (first_name, last_name) VALUES ('Fay', 'Diaz');   -- emp_id auto-assigned
SELECT last_insert_rowid() AS new_id, (SELECT emp_id FROM employees WHERE first_name = 'Fay') AS fay_id;

new_id | fay_id
-------+-------
6      | 6     
(1 row)



In [39]:
%%sql
CREATE TABLE t_int (id INT PRIMARY KEY, label TEXT);
INSERT INTO t_int (label) VALUES ('a'), ('b');
SELECT rowid, id, label FROM t_int ORDER BY rowid;   -- id is NULL: INT PRIMARY KEY is not a rowid alias

rowid | id   | label
------+------+------
1     | NULL | a    
2     | NULL | b    
(2 rows)



## 17. SQL execution order (why aliases fail in WHERE)
- You write: `SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT`.
- The engine logically runs: FROM (and JOINs) -> WHERE -> GROUP BY -> HAVING -> SELECT (compute columns/aliases) -> DISTINCT -> ORDER BY -> LIMIT/OFFSET.
- So `WHERE` runs before the SELECT list exists: standard SQL forbids aliases there. SQLite is lenient and allows it; do not rely on it.
- `ORDER BY` runs after SELECT, so it can use aliases.
- `WHERE` cannot use aggregates (`WHERE count(*) > 1` is an error); `HAVING` can. See 02-Querying/02-Aggregation-And-Grouping.

In [40]:
%%sql
-- portable: repeat the expression in WHERE, use the alias in ORDER BY
SELECT name, price * stock AS inventory_value
FROM products
WHERE price * stock > 100
ORDER BY inventory_value DESC, name;

name     | inventory_value
---------+----------------
Keyboard | 1247.5         
Notebook | 960.0          
(2 rows)



In [41]:
%%sql expect_error
SELECT dept, count(*) FROM employees WHERE count(*) > 1 GROUP BY dept;

ERROR: misuse of aggregate: count()



## 18. Comments, semicolons and formatting
- `-- comment` runs to the end of the line. `/* ... */` can span lines (no nesting).
- Statements end with `;`. A script can hold many statements.
- Keywords and function names are case-insensitive. Table and column names are case-insensitive too (but keep your style consistent).
- Strings use single quotes `'text'`. Double quotes are for identifiers (SQLite falls back to a string if no such column, a bug magnet).
- To put a quote inside a string, double it: `'it''s'`.

In [42]:
%%sql
/* multi-line comment:
   explain the query here */
SELECT 'it''s fine' AS quoted,   -- the doubled quote becomes one
       "name" AS ident           -- double quotes name a column
FROM products WHERE product_id = 1;

quoted    | ident   
----------+---------
it's fine | Keyboard
(1 row)



In [43]:
%%sql
-- the double-quote trap: no such column "nme", so SQLite quietly treats it as the string 'nme'
SELECT "nme" AS oops FROM products WHERE product_id = 1;

oops
----
nme 
(1 row)

